In [ ]:
import json
import gspread
from google.oauth2.service_account import Credentials
from playwright.async_api import async_playwright
import asyncio

In [ ]:
scope = ["https://www.googleapis.com/auth/spreadsheets","https://www.googleapis.com/auth/drive"]

# Buscar credenciais do JSON

creds = Credentials.from_service_account_file("credentials.json",scopes=scope)
client = gspread.authorize(creds)

# Nome da planilha/ABA
# Automatizar aqui

nome_planilha = "PAGAMENTOS EMN5 TESTES"
nome_aba = "EMN5"

In [ ]:
# Buscar a planilha
planilha = client.open(nome_planilha).worksheet(nome_aba)


CHROME_PROFILE = r"C:\Users\William Silva\AppData\Local\Google\Chrome\User Data\Profile 4"
# Automatizar o processo de coluna
def converter_coluna(coluna):
    coluna = coluna.upper()
    indice = 0
    for letra in coluna:
        indice = indice * 26 + ord(letra) - ord('A') + 1
    return indice

# Coluna
coluna = converter_coluna('r')
# Retornar valores
id = planilha.col_values(coluna)


In [ ]:
# Dicionário para receber as rotas
rotas = {}

# Pegar os IDS
pedido = id[1]

In [ ]:
url = f"https://envios.adminml.com/logistics/api/monitoring-route/route-detail?routeId={pedido}&siteId=MLB&isRouteTypeFlex=false"
print(pedido)

In [ ]:
# Sessão de login OKTA
# Realizar o login
async def main():
    print("loop1")
    async with async_playwright() as p:
        print("loop2")
        brower = await p.chromium.launch_persistent_context(user_data_dir= CHROME_PROFILE, headless=False)
        #context = brower.new_context()
        page = await brower.new_page()
        print("loop3")
        response = await page.goto(url, timeout=50000)
  #      page.goto(url)
   #     page.wait_for_timeout(50000)
        print("✅ Processo finalizado. Fechando navegador...")
        print("Autenticado")
        print(await response.text())
        # Retornar a request em Json
        if response.status != 200:
            print(await response.text())
            raise Exception(f"Erro: {response.text}")
        data = await response.json()
        print(f'Json retornado:')
        print(json.dumps(data, indent = 2, ensure_ascii=False))
        await brower.close()
if __name__ == "__main__":
    asyncio.run(main())